# 9 - Dashboards and deployment

The backend assembles its layers into a Panel app and models how it is exported or served. Most of this
needs a **live kernel/server** to interact; the cells build the objects and report what they produced.

- `layer_control` - a panel to toggle, reorder and restyle layers.
- `attribute_table` - a table of a vector layer's attributes; `linked=True` brushes with the map.
- `export_plan` / `save_app` - model and write a self-contained offline HTML.
- `serve` / `serve_plan` - model and launch a live server with per-session state.
- `share_url` - encode the view into a shareable URL.
- `live` / `push` - a streaming layer updated from Python.

**Setup** and data - the Heidelberg bird observations.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # deliberately silence all warnings to keep the rendered docs clean

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import geopandas as gpd
import numpy as np

birds = gpd.read_file(str(DATA / "heidelberg_birds_2023.geojson"))
birds["log_uncertainty"] = np.log10(birds["uncertainty_m"].clip(lower=1.0))
print("birds:", len(birds))

## `layer_control` - toggle / reorder / restyle

`layer_control()` builds a Panel control panel for the map's layers; `reorder=True` adds up/down buttons.

In [ ]:
m = InteractiveMap(crs=4326, title="layer control")
m.points(birds, column="log_uncertainty", size=6, cmap="viridis").colorbar()
panel = m.layer_control(reorder=True).layer_control_panel
print("layer control panel:", type(panel).__name__)
show(m)

## `attribute_table` - attributes, optionally linked

`attribute_table(features)` returns a Panel table of the non-geometry columns. `linked=True` makes it a two-way selection companion to the map.

In [ ]:
m = InteractiveMap(crs=4326)
m.points(birds, column="log_uncertainty", size=6, cmap="viridis")
table = m.attribute_table(birds)                 # read-only view
linked = m.attribute_table(birds, linked=True)   # two-way selection link
print("table:", type(table).__name__, "| linked table selectable:", linked.selectable)
show(m)

## `export_plan` and `save_app`

`export_plan` states how many widget-state pages an `embed=True` export would bake; `save_app` writes the self-contained HTML.

In [ ]:
import tempfile
from pathlib import Path as _Path

m = InteractiveMap(crs=4326).points(birds, column="log_uncertainty", size=6, cmap="viridis")
plan = m.export_plan(widgets=("cmap",))
print("export plan:", plan.total_states, "states | embeddable:", plan.embeddable)
out = m.save_app(str(_Path(tempfile.mkdtemp()) / "birds_app.html"), widgets=("cmap",))
print("wrote", _Path(out).name, "-", _Path(out).stat().st_size, "bytes")

## `serve_plan` and `serve`

`serve_plan` models the deployment; `serve` marks the app servable for `panel serve`, installing any per-session `onload` / `periodic` state. `start=True` launches a server.

In [ ]:
m = InteractiveMap(crs=4326).points(birds, column="log_uncertainty", size=6, cmap="viridis")
deployment = m.serve_plan(periodic=lambda: None, period_ms=1000)
print("deployment:", deployment.summary)
app = m.serve(widgets=("cmap",))   # servable viewable (start=False)
print("servable:", type(app).__name__)

## `share_url` - a reproducible link

`share_url()` encodes the figure spec into a URL blob and `from_share_url` rebuilds it. Only state that
resolves outside this process is encodable - the basemap, theme and extent, and path/URL-referenced data; an
in-memory data layer is an `object:` source that lives only here, so we share a basemap view.

In [ ]:
m = InteractiveMap(crs=3857, title="shared basemap view").basemap("CartoLight").theme("dark_minimal")
url = m.share_url()
print("share url length:", len(url), "chars")
rebuilt = InteractiveMap.from_share_url(url)
print("rebuilt layers:", len(rebuilt.layer_ids))

## `live` / `push` - a streaming layer

`live(kind=...)` registers a streaming layer backed by a HoloViews `Pipe`/`Buffer`; `push(layer_id, data)` sends new data so only that layer redraws (live only).

In [ ]:
m = InteractiveMap(crs=4326, title="streaming points")
m.live(kind="points", name="stream")
m.push("stream", np.array([[8.65, 49.40], [8.70, 49.42]]))
print("live layers:", m.layer_ids)
show(m)